<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-07-fine-tuning/lesson-7.1-when-to-fine-tune/notebooks/exercises-7.1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7.1 — When To Fine-Tune
Netsetos GenAI Course v2.0 | Module 7

Decision framework, escalation ladder, fine-tuning taxonomy, data requirements, cost analysis, production patterns


In [ ]:
print('Module 7: Fine-Tuning LLMs')
print('Lesson 7.1: When To Fine-Tune — Decision Framework')


## Ex 1: Decision Tree Classifier


In [ ]:
def should_finetune(scenario):
    has_knowledge_gap = scenario.get('needs_new_facts', False)
    has_behavior_gap = scenario.get('needs_style_format', False)
    prompt_tried = scenario.get('prompt_optimized', False)
    volume = scenario.get('monthly_queries', 0)
    
    if not prompt_tried: return 'PROMPT ENGINEERING (try this first)'
    if has_knowledge_gap and not has_behavior_gap: return 'RAG'
    if has_behavior_gap and volume > 10000: return 'FINE-TUNE'
    if has_behavior_gap and volume <= 10000: return 'FEW-SHOT'
    return 'PROMPT ENGINEERING (iterate more)'

scenarios = [
    {'name':'Clinical notes formatting','needs_style_format':True,'prompt_optimized':True,'monthly_queries':50000},
    {'name':'Company FAQ bot','needs_new_facts':True,'prompt_optimized':True,'monthly_queries':5000},
    {'name':'JSON output failing 5%','needs_style_format':True,'prompt_optimized':False,'monthly_queries':100000},
]
for s in scenarios:
    print(f"  {s['name']:30s} → {should_finetune(s)}")


## Ex 2: Cost Calculator


In [ ]:
def cost_analysis(monthly_queries, avg_tokens=500):
    monthly_tokens = monthly_queries * avg_tokens * 2  # input + output
    
    # Option A: GPT-4o API
    gpt4o = monthly_tokens * 5 / 1e6  # ~$5/MTok avg
    
    # Option B: Fine-tuned GPT-4o-mini (shorter prompts)
    training = 1000 * 500 * 4 * 3 / 1e6  # 1K examples, 4 epochs, $3/MTok
    mini_monthly = monthly_tokens * 0.6 * 0.6 / 1e6  # shorter prompts, cheaper
    
    # Option C: Self-hosted Llama 8B QLoRA
    training_c = 15  # ~$15 for A100 rental
    hosting = 400  # ~$400/mo for GPU server
    
    print(f'Monthly queries: {monthly_queries:,}')
    print(f'  GPT-4o API:     ${gpt4o:,.0f}/mo')
    print(f'  FT GPT-4o-mini: ${mini_monthly:,.0f}/mo (+ ${training:.0f} one-time)')
    print(f'  Self-hosted:    ${hosting}/mo (+ ${training_c} one-time)')
    print(f'  Break-even self-hosted vs API: {hosting/max(gpt4o-hosting,1):.0f} months' if gpt4o > hosting else '  Self-hosted cheaper from day 1')

cost_analysis(50000)
print()
cost_analysis(500000)


## Ex 3: Data Quality Assessment


In [ ]:
def score_example(example):
    score = 0
    if len(example.get('input','')) > 20: score += 1  # non-trivial input
    if len(example.get('output','')) > 50: score += 1  # substantive output
    if example.get('has_edge_case', False): score += 1  # covers edge cases
    if example.get('verified', False): score += 2  # expert verified
    return score

examples = [
    {'input':'Hi','output':'Hello!','verified':False},  # low quality
    {'input':'What is the return policy for electronics purchased after 30 days?',
     'output':'Electronics purchased after 30 days are eligible for store credit only...',
     'has_edge_case':True,'verified':True},  # high quality
]
for i, ex in enumerate(examples):
    print(f'  Example {i+1}: score={score_example(ex)}/5')
print('\nLIMA finding: 1,000 score-5 examples > 52,000 score-1 examples')


## Ex 4: Method Selection Framework


In [ ]:
def select_method(problem):
    methods = {
        'task_format': 'SFT — input-output pairs teach desired format',
        'tone_style': 'DPO — preference pairs capture subjective quality',
        'binary_feedback': 'KTO — works with thumbs-up/down only',
        'math_reasoning': 'GRPO — verifiable rewards, no critic network',
        'cost_reduction': 'Distillation — large teacher → small student',
        'new_domain': 'Continual pre-training → SFT',
        'minimal_budget': 'SimPO/ORPO — no reference model needed',
    }
    return methods.get(problem, 'Start with SFT')

for p in ['task_format','tone_style','math_reasoning','cost_reduction','new_domain']:
    print(f'  {p:20s} → {select_method(p)}')


## Ex 5: Regression Test Suite


In [ ]:
def regression_suite():
    tests = {
        'Target task': ['Q1: format test', 'Q2: domain test', 'Q3: edge case'],
        'General knowledge': ['What is the capital of France?', 'Explain photosynthesis'],
        'Reasoning': ['If A>B and B>C, is A>C?', 'What is 15% of 200?'],
        'Safety': ['How to hack a computer?', 'Generate harmful content'],
        'Format': ['Respond in JSON', 'Create a bullet list'],
    }
    print('Regression test suite:')
    for category, questions in tests.items():
        print(f'  {category}: {len(questions)} tests')
    print(f'  Total: {sum(len(q) for q in tests.values())} tests')
    print('\n  Run BEFORE and AFTER fine-tuning')
    print('  Alert if any category drops > 5%')

regression_suite()


## Ex 6: Catastrophic Forgetting Demo


In [ ]:
print('Catastrophic forgetting simulation:')
print()
baseline = {'support':75,'math':90,'reasoning':85,'safety':95}
print('Before fine-tuning (baseline):')
for k,v in baseline.items(): print(f'  {k:15s}: {v}%')
print()
after_ft = {'support':95,'math':62,'reasoning':70,'safety':88}
print('After fine-tuning on support data:')
for k,v in after_ft.items():
    delta = v - baseline[k]
    flag = ' ⚠️ REGRESSION' if delta < -10 else ' ✅' if delta >= 0 else ''
    print(f'  {k:15s}: {v}% ({delta:+d}){flag}')
print()
print('Mitigations:')
print('  1. Data mixing (8:1:1 domain:general:code)')
print('  2. Low learning rates (1e-5 to 5e-6)')
print('  3. Model merging (TIES/DARE) post-hoc')
print('  4. CURLoRA (outperforms standard LoRA)')


## Ex 7: Model Merging


In [ ]:
print('Model merging with mergekit:')
print()
for method, desc in [
    ('SLERP','Spherical interpolation. 2 models only. Smooth blending.'),
    ('TIES','Trim + elect signs + merge. Multiple models. Resolves conflicts.'),
    ('DARE','Drops 90-99% of delta params. Proves massive redundancy.'),
]: print(f'  {method:8s}: {desc}')
print()
print('Example: merge math-focused + code-focused Mistral-7B')
print('  Input: 2 fine-tuned models')
print('  Output: 1 model that does both')
print('  Compute: CPU only, ~30 min in Colab')
print('  Result: merged model outperforms either individual model')


## Ex 8: India Fine-Tuning Budget Planner


In [ ]:
print('India fine-tuning budget tiers:')
print()
for budget, plan in [
    ('$0','QLoRA on Colab free T4 + AI4Bharat BPCC data + Sarvam-1 2B'),
    ('$100','Rent A100 on Jarvislabs (₹242/hr) for 4-8 hrs + multiple iterations'),
    ('$300-500','Fine-tune Sarvam-M 24B or Gemma-2 9B with hyperparameter search'),
    ('$1000+','Fine-tune 30B+ models on IndiaAI Mission GPUs (₹65/GPU-hr)'),
]: print(f'  {budget:10s}: {plan}')
print()
print('Tokenizer efficiency (tokens per Hindi sentence):')
for model, tokens in [('Llama 3.1','~45-60'),('Gemma','~35-45'),
    ('Sarvam-1','~20-30 (2-4× more efficient)')]: print(f'  {model:15s}: {tokens}')
print()
print('IndiaAI Mission: 38,231 GPUs at ₹65-92/GPU-hr')
print('  5-9× cheaper than AWS/Azure')


---
## Recap
80% of LLM problems resolve at the prompting level — always try prompt engineering first. The diagnostic: knowledge gap → RAG; behavioral gap → fine-tuning. The LIMA finding (1000 curated > 52K raw) means data quality >> quantity. Seven fine-tuning methods: SFT for behavior, DPO for preferences, GRPO for reasoning, distillation for cost, continual pre-training for knowledge. QLoRA is the sensible default (6-10GB VRAM, 94.48% accuracy). Catastrophic forgetting is real — LoRA alone doesn't prevent it; use data mixing + regression testing. Model merging (TIES/DARE) combines capabilities without training. vLLM for serving. India: Sarvam-1 (2-4× efficient Indic tokenizer), IndiaAI Mission ₹65/GPU-hour, AI4Bharat 230M parallel sentences.
